<a href="https://colab.research.google.com/github/filipsajtlava/dspracticum2025-tismaci/blob/homework7/homeworks/hw7/homework07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

In [ ]:
train_data_path = "http://raw.githubusercontent.com/filipsajtlava/dspracticum2025-tismaci/refs/heads/master/homeworks/hw7/appartments_train.csv"
test_data_path = "https://raw.githubusercontent.com/filipsajtlava/dspracticum2025-tismaci/refs/heads/master/homeworks/hw7/appartments_test.csv"

In [ ]:
try:
    df_train = pd.read_csv(train_data_path)
    df_test = pd.read_csv(test_data_path)
    print("Data byla načtena.")
except Exception as e:
    print(f"Chyba: {e}")

Data byla načtena.


In [ ]:
categorical = ['layout', 'construction', 'condition', 'ownership', 'elevator', 'address']
not_using = ['text', 'first_seen', 'last_seen', 'price']
using = [col for col in df_train.columns if col not in not_using]

X_train = df_train[using].copy()
y_train = df_train['price'].copy()
X_test = df_test[using].copy()

for col in categorical:
    if col in X_train.columns:
        X_train[col] = X_train[col].fillna('N/A').astype('str')
        X_test[col] = X_test[col].fillna('N/A').astype('str')


In [ ]:
model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.04,
    depth=7,
    loss_function='MAPE',
    eval_metric='MAPE',
    random_seed=128,
    verbose=300,
    cat_features=categorical
)

In [ ]:
model.fit(X_train, y_train, early_stopping_rounds=300)

0:	learn: 0.2713943	total: 66.1ms	remaining: 1m 39s
300:	learn: 0.2131558	total: 5.26s	remaining: 21s
600:	learn: 0.2040541	total: 10s	remaining: 15s
900:	learn: 0.1992164	total: 14.4s	remaining: 9.57s
1200:	learn: 0.1947655	total: 20.5s	remaining: 5.12s
1499:	learn: 0.1915802	total: 25.3s	remaining: 0us


In [ ]:
feature_importances = pd.Series(model.get_feature_importance(), index=X_train.columns)
top_features = feature_importances.sort_values(ascending=False).head(15)
print(f'Dulezite promenne: {top_features}.')
flop_features = feature_importances[feature_importances < 0.1]
if not flop_features.empty:
    print(f"Málo důležité proměnné ({len(flop_features)}):")
    print(flop_features)

Dulezite promenne: area                               24.428789
layout                             16.705653
construction                       14.721440
condition                          14.603320
ownership                           8.650047
gps_lon                             5.190385
elevator                            3.081283
balcony_area                        3.009621
gps_lat                             2.448742
poi_doctors_nearest                 1.627158
poi_school_kindergarten_nearest     0.953655
id                                  0.938376
poi_leisure_time_nearest            0.831178
poi_grocery_nearest                 0.816375
poi_restaurant_nearest              0.773172
dtype: float64.
Málo důležité proměnné (9):
address                          0.000000
garden_area                      0.000000
parking                          0.094121
poi_doctors_count                0.000000
poi_leisure_time_count           0.057813
poi_school_kindergarten_count    0.078964
poi_transp

In [ ]:
categorical = ['layout', 'construction', 'condition', 'ownership', 'elevator']
not_using = ['text', 'first_seen', 'last_seen', 'price']
flop = flop_features.index
flop = flop.tolist()
using = [col for col in df_train.columns if col not in not_using + flop]

X_train = df_train[using].copy()
y_train = df_train['price'].copy()
X_test = df_test[using].copy()

for col in categorical:
    if col in X_train.columns:
        X_train[col] = X_train[col].fillna('N/A').astype('str')
        X_test[col] = X_test[col].fillna('N/A').astype('str')

In [ ]:
model = CatBoostRegressor(
    iterations=40000,
    learning_rate=0.02,
    depth=7,
    loss_function='MAPE',
    eval_metric='MAPE',
    random_seed=128,
    verbose=300,
    cat_features=categorical
)

In [ ]:
model.fit(X_train, y_train, early_stopping_rounds=300)

0:	learn: 0.2716674	total: 26ms	remaining: 17m 19s
300:	learn: 0.2182604	total: 3.71s	remaining: 8m 9s
600:	learn: 0.2079241	total: 7.37s	remaining: 8m 3s
900:	learn: 0.2034306	total: 12.1s	remaining: 8m 44s
1200:	learn: 0.2009574	total: 16.1s	remaining: 8m 38s
1500:	learn: 0.1980499	total: 19.8s	remaining: 8m 27s
1800:	learn: 0.1956282	total: 24.7s	remaining: 8m 44s
2100:	learn: 0.1941578	total: 28.7s	remaining: 8m 37s
2400:	learn: 0.1921312	total: 32.5s	remaining: 8m 29s
2700:	learn: 0.1906732	total: 37.5s	remaining: 8m 37s
3000:	learn: 0.1892847	total: 41.4s	remaining: 8m 29s
3300:	learn: 0.1877876	total: 45.1s	remaining: 8m 21s
3600:	learn: 0.1865746	total: 50.3s	remaining: 8m 27s
3900:	learn: 0.1854500	total: 54.4s	remaining: 8m 23s
4200:	learn: 0.1842979	total: 58.1s	remaining: 8m 14s
4500:	learn: 0.1835184	total: 1m 3s	remaining: 8m 17s
4800:	learn: 0.1827006	total: 1m 6s	remaining: 8m 10s
5100:	learn: 0.1816352	total: 1m 10s	remaining: 8m 3s
5400:	learn: 0.1808122	total: 1m 15s

In [ ]:
y_pred_test = model.predict(X_test)

In [ ]:
prediction = pd.DataFrame({
    'id': df_test['id'],
    'price': y_pred_test.round(0).astype(int)
})
prediction.to_csv('predikce.csv')